# PowerGuard AI — Notebook 02: Model Training & Evaluation

This notebook trains both ML models, evaluates them on held-out test sets,
and visualises the results.  All numbers are real — produced by running the
training scripts on the actual processed data.

## Two independent models

| Model | Task | Dataset | Split strategy |
|-------|------|---------|----------------|
| **Model A** | Transformer risk class (LOW/MEDIUM/HIGH) | 470 DGA oil-analysis samples | Stratified random 70/15/15 |
| **Model B** | Weather outage severity (LOW/MEDIUM/HIGH) | 33,139 aggregated events | Chronological (pre-2021 train, 2021+ test) |

> **Why two models?**  
> The datasets share no common key — they cannot be joined.  
> At inference time their probability outputs are fused with a weighted sum:
> `combined_risk = 0.6 × P_A(HIGH) + 0.3 × P_B(HIGH) + 0.1 × criticality`

**Run from the repo root before executing this notebook:**
```bash
python -m src.data.pipeline          # generate processed CSVs
python -m src.ml.models.train_model_a
python -m src.ml.models.train_model_b
```

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

# Ensure repo root is on sys.path when running from notebooks/
REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

ARTIFACTS = REPO_ROOT / 'src' / 'ml' / 'models' / 'artifacts'

print('Repo root  :', REPO_ROOT)
print('Artifacts  :', ARTIFACTS)
print('Artifacts exist:', ARTIFACTS.exists())

## 1. Load saved results

In [ ]:
with open(ARTIFACTS / 'model_a_results.json') as f:
    res_a = json.load(f)['model_a_transformer_risk']

with open(ARTIFACTS / 'model_b_results.json') as f:
    res_b = json.load(f)['model_b_weather_severity']

print('Model A results loaded. Keys:', list(res_a.keys()))
print('Model B results loaded. Keys:', list(res_b.keys()))

## 2. Model A — Transformer Risk Classifier

### 2.1 Dataset summary

In [ ]:
si_a = res_a['split_info']
print(f"Total samples : {si_a['n_train'] + si_a['n_val'] + si_a['n_test']}")
print(f"Train         : {si_a['n_train']}  classes={si_a['train_class_dist']}")
print(f"Val           : {si_a['n_val']}   classes={si_a['val_class_dist']}")
print(f"Test          : {si_a['n_test']}   classes={si_a['test_class_dist']}")
print(f"\nClass labels  : 0=LOW  1=MEDIUM  2=HIGH")

### 2.2 Cross-validation results (5-fold stratified on training set)

In [ ]:
cv_a = res_a['cv']
cv_df_a = pd.DataFrame({
    'Model':        ['LR Baseline', 'RF Main'],
    'CV F1-weighted (mean)': [cv_a['lr_cv_f1_weighted_mean'], cv_a['rf_cv_f1_weighted_mean']],
    'CV F1-weighted (±std)': [cv_a['lr_cv_f1_weighted_std'],  cv_a['rf_cv_f1_weighted_std']],
    'CV Accuracy (mean)':   [cv_a['lr_cv_accuracy_mean'],    cv_a['rf_cv_accuracy_mean']],
})
cv_df_a = cv_df_a.set_index('Model').round(4)
print('Model A — 5-fold CV on training set (n=329):')
print(cv_df_a.to_string())

### 2.3 Test-set evaluation (held-out, never seen during training)

In [ ]:
test_rf_a  = res_a['test_rf']
test_lr_a  = res_a['test_lr']

print('=== Random Forest (SELECTED MODEL) ===')
print(test_rf_a['classification_report_str'])
print(f"ROC-AUC (macro OVR): {test_rf_a['roc_auc_macro']}")

print('\n=== Logistic Regression (Baseline) ===')
print(test_lr_a['classification_report_str'])
print(f"ROC-AUC (macro OVR): {test_lr_a['roc_auc_macro']}")

### 2.4 Confusion matrix — RF on test set

In [ ]:
cm_a   = np.array(test_rf_a['confusion_matrix'])
labels = test_rf_a['confusion_matrix_labels']

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm_a, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels, ax=ax,
    linewidths=0.5, linecolor='white'
)
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('True',      fontsize=11)
ax.set_title('Model A — RF Confusion Matrix (Test Set, n=71)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'model_a_confusion_matrix.png', dpi=120)
plt.show()
print('Saved to artifacts/model_a_confusion_matrix.png')

### 2.5 Feature importances — RF

In [ ]:
fi_a = pd.Series(res_a['feature_importances']).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#d73027' if fi_a[f] >= fi_a.quantile(0.75) else '#4575b4' for f in fi_a.index]
ax.barh(fi_a.index, fi_a.values, color=colors, edgecolor='white', height=0.7)
ax.set_xlabel('Mean Decrease in Impurity', fontsize=11)
ax.set_title('Model A — RF Feature Importances (all 20 features)', fontsize=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.tight_layout()
plt.savefig(ARTIFACTS / 'model_a_feature_importances.png', dpi=120)
plt.show()
print('\nTop 5 features:')
for feat, imp in fi_a.sort_values(ascending=False).head(5).items():
    print(f'  {feat:<35} {imp:.4f} ({imp*100:.1f}%)')

### 2.6 Model A — Summary scorecard

| Metric | LR Baseline | **RF (selected)** |
|--------|-------------|-------------------|
| CV F1-weighted | 0.7520 ± 0.0330 | **0.8152 ± 0.0461** |
| Test Accuracy | 0.5775 | **0.8310** |
| Test F1 macro | 0.6105 | **0.8480** |
| Test F1 weighted | 0.5866 | **0.8320** |
| Test ROC-AUC | 0.8317 | **0.9132** |

**Interpretation:**  
The RF model substantially outperforms LR on every metric.  
ROC-AUC of **0.9132** confirms the model separates HIGH-risk transformers
from LOW/MEDIUM reliably.  Top signals are `hydrogen_ppm` (15.5%) and
`dbds_mg_kg` (10.5%), consistent with IEC 60599 DGA fault detection practice.

---

## 3. Model B — Weather Outage Severity Classifier

### 3.1 Dataset summary

In [ ]:
si_b = res_b['split_info']
print(f"Total samples : {si_b['n_train'] + si_b['n_test']}")
print(f"Cutoff        : {si_b['train_cutoff']}  (chronological split)")
print(f"Train         : {si_b['n_train']}  classes={si_b['train_class_dist']}")
print(f"Test          : {si_b['n_test']}   classes={si_b['test_class_dist']}")

train_dist = np.array(si_b['train_class_dist'])
print(f"\nTrain class %  : LOW={train_dist[0]/train_dist.sum()*100:.1f}%  "
      f"MEDIUM={train_dist[1]/train_dist.sum()*100:.1f}%  "
      f"HIGH={train_dist[2]/train_dist.sum()*100:.1f}%")
print('→ Severe imbalance — class_weight=balanced applied to the RF.')

### 3.2 Cross-validation results (5-fold on 10,000-sample sub-set)

In [ ]:
cv_b = res_b['cv']
cv_df_b = pd.DataFrame({
    'Model':            ['LR Baseline', 'RF Main'],
    'CV F1-weighted':   [cv_b['lr_cv_f1_weighted_mean'], cv_b['rf_cv_f1_weighted_mean']],
    '±std':             [cv_b['lr_cv_f1_weighted_std'],  cv_b['rf_cv_f1_weighted_std']],
    'CV F1-macro':      [cv_b['lr_cv_f1_macro_mean'],    cv_b['rf_cv_f1_macro_mean']],
})
cv_df_b = cv_df_b.set_index('Model').round(4)
print(f"Model B — 5-fold CV on {cv_b['cv_sample_size']}-sample sub-set of training data:")
print(cv_df_b.to_string())

### 3.3 Test-set evaluation

In [ ]:
test_rf_b  = res_b['test_rf']
test_lr_b  = res_b['test_lr']

print('=== Random Forest (SELECTED MODEL) ===')
print(test_rf_b['classification_report_str'])
print(f"ROC-AUC (macro OVR): {test_rf_b['roc_auc_macro']}")

print('\n=== Logistic Regression (Baseline) ===')
print(test_lr_b['classification_report_str'])
print(f"ROC-AUC (macro OVR): {test_lr_b['roc_auc_macro']}")

### 3.4 Confusion matrix — RF on test set

In [ ]:
cm_b    = np.array(test_rf_b['confusion_matrix'])
labels  = test_rf_b['confusion_matrix_labels']

# Normalise rows to show recall per class
cm_b_norm = cm_b.astype(float) / cm_b.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

sns.heatmap(cm_b,      annot=True, fmt='d',   cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0],
            linewidths=0.5, linecolor='white')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_title('Raw counts')

sns.heatmap(cm_b_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[1],
            linewidths=0.5, linecolor='white', vmin=0, vmax=1)
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_title('Row-normalised (recall)')

fig.suptitle('Model B — RF Confusion Matrix (Test Set, n=15,177)', fontsize=12)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'model_b_confusion_matrix.png', dpi=120)
plt.show()
print('Saved to artifacts/model_b_confusion_matrix.png')

### 3.5 Feature importances — RF

In [ ]:
fi_b = pd.Series(res_b['feature_importances']).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(6, 4))
colors = ['#d73027' if fi_b[f] >= fi_b.quantile(0.6) else '#4575b4' for f in fi_b.index]
ax.barh(fi_b.index, fi_b.values, color=colors, edgecolor='white', height=0.6)
ax.set_xlabel('Mean Decrease in Impurity', fontsize=11)
ax.set_title('Model B — RF Feature Importances', fontsize=12)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.tight_layout()
plt.savefig(ARTIFACTS / 'model_b_feature_importances.png', dpi=120)
plt.show()
print('\nAll feature importances:')
for feat, imp in fi_b.sort_values(ascending=False).items():
    print(f'  {feat:<20} {imp:.4f} ({imp*100:.1f}%)')

### 3.6 Model B — Summary scorecard

| Metric | LR Baseline | **RF (selected)** |
|--------|-------------|-------------------|
| CV F1-weighted | 0.5339 ± 0.0135 | **0.6862 ± 0.0128** |
| CV F1-macro    | 0.3484 | **0.4992** |
| Test Accuracy  | 0.2483 | **0.6107** |
| Test F1 macro  | 0.2268 | **0.4451** |
| Test F1 weighted | 0.3316 | **0.6150** |
| Test ROC-AUC   | 0.6204 | **0.6589** |

**Interpretation:**  
Weather features alone are moderate predictors of outage severity.  
The RF ROC-AUC of **0.6589** is above random (0.5) but well below Model A.  
This is expected — outage severity depends strongly on grid topology,
maintenance history, and line capacity, none of which are captured in
weather-only data.  The model still provides a useful risk *modifier* for
the fusion score, boosting alarm sensitivity during dangerous weather.

---

## 4. Model comparison — side-by-side

In [ ]:
metrics = ['accuracy', 'f1_macro', 'f1_weighted', 'roc_auc']
comparison = pd.DataFrame({
    'Metric': ['Test Accuracy', 'Test F1-macro', 'Test F1-weighted', 'Test ROC-AUC (macro)'],
    'Model A — Transformer RF': [
        test_rf_a['accuracy'],
        test_rf_a['classification_report']['macro avg']['f1-score'],
        test_rf_a['classification_report']['weighted avg']['f1-score'],
        test_rf_a['roc_auc_macro'],
    ],
    'Model B — Weather RF': [
        test_rf_b['accuracy'],
        test_rf_b['classification_report']['macro avg']['f1-score'],
        test_rf_b['classification_report']['weighted avg']['f1-score'],
        test_rf_b['roc_auc_macro'],
    ],
}).set_index('Metric').round(4)

print(comparison.to_string())

# Bar chart comparison
ax = comparison.plot.bar(figsize=(8, 4), rot=15, ylim=(0, 1.0),
                         color=['#3b82d4', '#7c5cd8'], edgecolor='white')
ax.set_ylabel('Score')
ax.set_title('Model A vs Model B — Test Set Metrics', fontsize=12)
ax.legend(loc='lower right')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
plt.tight_layout()
plt.savefig(ARTIFACTS / 'model_comparison.png', dpi=120)
plt.show()
print('Saved to artifacts/model_comparison.png')

## 5. Inference fusion formula

At prediction time the two models output class probabilities.
These are fused into a single `combined_risk_score ∈ [0, 1]`:

```
combined_risk = 0.60 × P_A(HIGH) + 0.30 × P_B(HIGH) + 0.10 × criticality_score
```

Where:
- `P_A(HIGH)` = Model A probability of transformer being HIGH-risk
- `P_B(HIGH)` = Model B probability of HIGH outage severity for current weather
- `criticality_score` = normalised asset criticality (voltage level, customers served)

The score is then bucketed:
- **≥ 0.65** → CRITICAL (immediate dispatch)
- **0.40 – 0.64** → HIGH (schedule within 48 h)
- **0.20 – 0.39** → MEDIUM (plan next maintenance cycle)
- **< 0.20** → LOW (routine monitoring)

In [ ]:
# Demo: compute combined_risk for a range of P_A, P_B values
p_a_vals = np.linspace(0, 1, 100)
p_b_fixed = 0.4    # moderate weather risk
crit_fixed = 0.5   # medium criticality

combined = 0.6 * p_a_vals + 0.3 * p_b_fixed + 0.1 * crit_fixed

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(p_a_vals, combined, color='#3b82d4', lw=2)
ax.axhline(0.65, color='#d73027', ls='--', lw=1.2, label='CRITICAL threshold (0.65)')
ax.axhline(0.40, color='#fc8d59', ls='--', lw=1.2, label='HIGH threshold (0.40)')
ax.axhline(0.20, color='#fee090', ls='--', lw=1.2, label='MEDIUM threshold (0.20)')
ax.fill_between(p_a_vals, combined, 0.65, where=(combined >= 0.65),
                alpha=0.15, color='#d73027', label='CRITICAL zone')
ax.set_xlabel('P_A (Model A HIGH-risk probability)', fontsize=10)
ax.set_ylabel('combined_risk_score', fontsize=10)
ax.set_title(f'Fusion score vs P_A  [P_B=0.4, criticality=0.5]', fontsize=11)
ax.legend(fontsize=8, loc='upper left')
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(ARTIFACTS / 'fusion_score_demo.png', dpi=120)
plt.show()

---

## 6. Artifact inventory

In [ ]:
print('Files in artifacts/ directory:')
for p in sorted(ARTIFACTS.iterdir()):
    size_kb = p.stat().st_size / 1024
    print(f'  {p.name:<45}  {size_kb:>8.1f} KB')

## 7. Next steps

The ML layer is now complete.  The next phase builds the **Flask backend** that
wraps these models behind a REST API:

| Endpoint | Method | Description |
|----------|--------|-------------|
| `/api/v1/assets` | GET | List all transformer assets |
| `/api/v1/assets/<id>/risk` | GET | Predict risk for one transformer |
| `/api/v1/dashboard/risk-summary` | GET | Aggregated risk counts for dashboard |
| `/api/v1/maintenance/plan` | GET | Ranked maintenance plan |
| `/api/v1/weather/current` | GET | Current weather severity by region |

The SQLite database will store:
- `asset` — transformer registry (id, name, location, voltage_kv, customers_served)
- `sensor_reading` — DGA measurements per asset over time
- `risk_score` — stored inference results
- `maintenance_order` — generated work orders
- `weather_event` — cached weather severity records